# AI Model Experiment & Evaluation

Notebook ini membahas eksperimen membandingkan dua pendekatan untuk **sentiment analysis ulasan produk e-commerce**:

1. **Model klasik**: TF-IDF + Logistic Regression (Scikit-learn)

2. **LLM API**: prompt zero-shot melalui Gemini API (dengan fallback OpenRouter)

Kedua pendekatan dievaluasi pada **test set yang sama** agar perbandingannya adil.

---

## 1. Problem Statement

**Objective:** tim produk e-commerce ingin membangun fitur otomatis yang mengklasifikasikan sentimen ulasan pelanggan (positif/negatif) di halaman produk. Sebelum tim memutuskan pendekatan mana yang akan dipakai di sistem produksi, dilakukan eksperimen awal untuk membandingkan dua kandidat pendekatan: model klasik yang dilatih sendiri vs LLM API berbasis prompt.

**Target/label:** kolom `sentiment` dengan dua kelas, `positif` dan `negatif`.

**Batasan dan asumsi:**

- Dataset yang digunakan hanya `customer_reviews_sentiment.csv` (200 baris), sesuai ketentuan tugas.

- Ulasan berbahasa Indonesia dan pendek (43-71 karakter).

- Label pada dataset dianggap benar.

- Test set yang sama digunakan untuk kedua pendekatan.

- (Temuan eksplorasi) Dataset banyak berisi template review yang diulang, sehingga split dilakukan per template unik agar tidak terjadi kebocoran data.

## 2. Import Library & Load Dataset

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
df = pd.read_csv('../data/customer_reviews_sentiment.csv')
print('Shape:', df.shape)
df.head()

Shape: (200, 4)


,review_id,product_name,review_text,sentiment
0,1,Kemeja Flanel,Pelayanan lambat dan tidak responsif saat diko...,negatif
1,2,Case HP,"Puas banget belanja disini, proses cepat dan b...",positif
2,3,Rice Cooker,"Terima kasih seller, barangnya awet dan sesuai...",positif
3,4,Case HP,Warna produk berbeda jauh dari foto di listing.,negatif
4,5,Headset Bluetooth,"Terima kasih seller, barangnya awet dan sesuai...",positif


## 3. Menyiapkan Train/Test Split

> **Temuan penting dari eksplorasi:** dari 200 baris, hanya **40 template review unik**; sisanya merupakan pengulangan dari template yang sama. Jika split dilakukan mentah-mentah, template yang sama dapat muncul di train *dan* test, sehingga model hanya menghafal daripada belajar. Karena itu, split dilakukan **setelah menghapus duplikat `review_text`**, lalu test set dipakai untuk kedua pendekatan.

In [2]:
# Cek duplikasi review_text
n_dupe = df.duplicated(subset='review_text').sum()
print(f'Baris duplikat (berdasarkan review_text): {n_dupe}')


# Ambil template unik agar split tidak bocor
df_uniq = df.drop_duplicates(subset='review_text').reset_index(drop=True)
print(f'Template unik: {len(df_uniq)} baris')
print(df_uniq['sentiment'].value_counts())
X = df_uniq['review_text']
y = df_uniq['sentiment']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train size:', len(X_train))
print('Test size:', len(X_test))
print('Distribusi label test:', y_test.value_counts().to_dict())

Baris duplikat (berdasarkan review_text): 160
Template unik: 40 baris
sentiment
negatif    20
positif    20
Name: count, dtype: int64
Train size: 32
Test size: 8
Distribusi label test: {'negatif': 4, 'positif': 4}


## 4. Pendekatan 1: Model Klasik (Scikit-learn)

Pendekatan ini menggunakan TF-IDF untuk mengubah teks menjadi angka, lalu Logistic Regression untuk klasifikasi. Ini merupakan pendekatan yang paling umum untuk teks pendek seperti ini.

### 4.1 Preprocessing & Feature Extraction

In [3]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print('Dimensi fitur TF-IDF:', X_train_vec.shape)

Dimensi fitur TF-IDF: (32, 131)


### 4.2 Training Model

> **Mengapa Logistic Regression?** Dataset ini merupakan klasifikasi biner (positif/negatif) dengan teks pendek, dan kerangkanya memang sudah disediakan di starter notebook (`LogisticRegression` dari Scikit-learn). Untuk kasus seperti ini, model linear seperti LR sudah cukup: cepat dilatih, ringan, dan mudah diinterpretasi, sehingga dapat menjadi baseline yang adil sebelum mencoba metode yang lebih kompleks (SVM, Random Forest, atau model bahasa). Pada dataset sekecil ini (32 data train), model yang lebih kompleks biasanya belum memberikan keuntungan berarti.

In [4]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)
print('Model dilatih.')

Model dilatih.


### 4.3 Prediksi pada Test Set

In [5]:
y_pred_classic = clf.predict(X_test_vec)
print(y_pred_classic.tolist())

['negatif', 'negatif', 'positif', 'positif', 'negatif', 'positif', 'negatif', 'positif']


## 5. Pendekatan 2: LLM API

### 5.1 Setup API

API key dibaca dari environment variable melalui file `.env` (dengan `python-dotenv`), **tidak di-hardcode**. Notebook ini mendukung **dua jalur**: jika `GEMINI_API_KEY` tersedia, digunakan Gemini langsung; jika tidak (atau terkena pembatasan region), fallback ke OpenRouter.

Catatan model:

- Gemini: `gemini-3.6-flash` (versi terbaru yang tersedia; `gemini-2.0-flash` di starter notebook sudah tidak tersedia).

- Fallback OpenRouter: `deepseek/deepseek-v4-flash-0731` (lebih murah dan output-nya stabil).

In [6]:
import json
import urllib.request
import time
import os
from dotenv import load_dotenv


# Baca file .env di direktori kerja (jika ada) agar API key mudah dikelola
load_dotenv()
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', '')


# Model untuk tiap jalur:
# - Gemini langsung: gemini-3.6-flash (versi terbaru yang tersedia)
# - Fallback OpenRouter: deepseek-v4-flash-0731 (murah dan output bersih)
GEMINI_MODEL = 'gemini-3.6-flash'
OPENROUTER_MODEL = 'deepseek/deepseek-v4-flash-0731'
print('GEMINI_API_KEY tersedia:', bool(GEMINI_API_KEY))
print('OPENROUTER_API_KEY tersedia:', bool(OPENROUTER_API_KEY))

GEMINI_API_KEY tersedia: True
OPENROUTER_API_KEY tersedia: True


### 5.2 Merancang Prompt

Prompt menggunakan pendekatan **zero-shot**: model langsung diminta mengklasifikasikan tanpa contoh sebelumnya. Alasannya, kelasnya hanya dua dan teksnya pendek, sehingga zero-shot sudah cukup dan paling murah. Sistem prompt dibuat tegas ("jawab hanya satu kata") agar output mudah dinormalisasi.

Catatan: suhu (`temperature`) diatur ke **0.2** (rendah, agar output konsisten, karena tugas klasifikasi ini membutuhkan determinisme), bukan kreativitas.

In [7]:
SYSTEM_PROMPT = (
    'Kamu adalah classifier sentimen untuk ulasan produk e-commerce berbahasa Indonesia. '
    'Jawab HANYA dengan satu kata: positif atau negatif. Tanpa tanda baca, tanpa penjelasan.'
)


def build_user_prompt(review_text):
    return f'Ulasan: "{review_text}"\nSentimen:'


def normalize_llm_output(raw):
    """Bersihkan output LLM menjadi salah satu dari: positif / negatif."""
    raw = (raw or '').strip().lower()
    clean = ''.join(ch for ch in raw if ch.isalpha() or ch.isspace()).strip()
    words = clean.split()
    if words and words[0].startswith('positif'):
        return 'positif'
    if words and words[0].startswith('negatif'):
        return 'negatif'
    # fallback: cek kata kunci sentimen apabila output tidak dikenali
    if any(k in raw for k in ['puas', 'bagus', 'manta', 'recommend', 'suka', 'terima kasih', 'cepat', 'awet', 'rapi', 'sesuai', 'original']):
        return 'positif'
    if any(k in raw for k in ['lambat', 'buruk', 'jelek', 'kecewa', 'rusak', 'palsu', 'menyesatkan', 'komplain', 'mahal', 'beda']):
        return 'negatif'
    return 'negatif'  # default label mayoritas

### 5.3 Menjalankan Prediksi pada Test Set

Loop memanggil LLM untuk setiap review. Output dinormalisasi (lowercase, strip, ambil kata pertama) sebelum dibandingkan dengan label asli.

In [ ]:
def classify_llm(review_text):
    # Jalur 1: Gemini langsung (jika key tersedia dan region mendukung)
    if GEMINI_API_KEY:
        try:
            from google import genai
            from google.genai import types
            client = genai.Client(api_key=GEMINI_API_KEY)
            resp = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=build_user_prompt(review_text),
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0.2,
                ),
            )
            return normalize_llm_output(resp.text)
        except Exception as e:
            print('Gemini gagal, fallback ke OpenRouter:', str(e)[:120])
            # lanjut ke jalur fallback
    # Jalur 2: OpenRouter (chat completions)
    if OPENROUTER_API_KEY:
        payload = {
            'model': OPENROUTER_MODEL,
            'messages': [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': build_user_prompt(review_text)},
            ],
            'temperature': 0.2,
            'max_tokens': 200,
        }
        req = urllib.request.Request(
            'https://openrouter.ai/api/v1/chat/completions',
            data=json.dumps(payload).encode(),
            headers={
                'Authorization': f'Bearer {OPENROUTER_API_KEY}',
                'Content-Type': 'application/json',
            },
        )
        with urllib.request.urlopen(req, timeout=60) as resp:
            d = json.loads(resp.read())
        content = (d['choices'][0]['message'].get('content') or '').strip()
        return normalize_llm_output(content)
    raise RuntimeError('Tidak ada API key tersedia (GEMINI_API_KEY / OPENROUTER_API_KEY)')
# Jalankan prediksi pada test set
y_pred_llm = []


for i, review in enumerate(X_test):
    pred = classify_llm(review)
    y_pred_llm.append(pred)
    print(f'[{i+1}/{len(X_test)}] {review[:45]!r} -> {pred}')
    time.sleep(0.3)  # jeda singkat untuk menghindari rate limit
print('\nPrediksi LLM:', y_pred_llm)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[1/8] 'Respon penjual lambat dan terkesan tidak pedu' -> negatif
Gemini gagal, fallback ke OpenRouter: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
[2/8] 'Komplain tidak ditanggapi sama sekali oleh pe' -> negatif
Gemini gagal, fallback ke OpenRouter: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
[3/8] 'Harga worth it dengan kualitas yang didapat, ' -> positif
Gemini gagal, fallback ke OpenRouter: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
[4/8] 'Pelayanan ramah dan responsif, barang sampai ' -> negatif
Gemini gagal, fallback ke OpenRouter: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
[5/8] 'Pengiriman lambat sekali, sudah dua minggu be' -> negatif
Gemini gagal, fallback ke OpenRouter: 4

## 6. Evaluasi dan Perbandingan

Metrik dihitung dengan `pos_label='positif'`.

### 6.1 Evaluasi Model Klasik

In [9]:
print('=== Model Klasik (Scikit-learn) ===')
print('Accuracy :', round(accuracy_score(y_test, y_pred_classic), 4))
print('Precision:', round(precision_score(y_test, y_pred_classic, pos_label='positif'), 4))
print('Recall   :', round(recall_score(y_test, y_pred_classic, pos_label='positif'), 4))
print('F1       :', round(f1_score(y_test, y_pred_classic, pos_label='positif'), 4))
print()
print('Confusion Matrix (baris=asli, kolom=prediksi): [positif, negatif]')
print(confusion_matrix(y_test, y_pred_classic, labels=['positif', 'negatif']))
print()
print(classification_report(y_test, y_pred_classic, target_names=['negatif', 'positif'], digits=4))

=== Model Klasik (Scikit-learn) ===
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1       : 1.0

Confusion Matrix (baris=asli, kolom=prediksi): [positif, negatif]
[[4 0]
 [0 4]]

              precision    recall  f1-score   support

     negatif     1.0000    1.0000    1.0000         4
     positif     1.0000    1.0000    1.0000         4

    accuracy                         1.0000         8
   macro avg     1.0000    1.0000    1.0000         8
weighted avg     1.0000    1.0000    1.0000         8



### 6.2 Evaluasi LLM API

In [10]:
print('=== LLM API ===')
print('Accuracy :', round(accuracy_score(y_test, y_pred_llm), 4))
print('Precision:', round(precision_score(y_test, y_pred_llm, pos_label='positif'), 4))
print('Recall   :', round(recall_score(y_test, y_pred_llm, pos_label='positif'), 4))
print('F1       :', round(f1_score(y_test, y_pred_llm, pos_label='positif'), 4))
print()
print('Confusion Matrix (baris=asli, kolom=prediksi): [positif, negatif]')
print(confusion_matrix(y_test, y_pred_llm, labels=['positif', 'negatif']))
print()
print(classification_report(y_test, y_pred_llm, target_names=['negatif', 'positif'], digits=4))

=== LLM API ===
Accuracy : 0.875
Precision: 1.0
Recall   : 0.75
F1       : 0.8571

Confusion Matrix (baris=asli, kolom=prediksi): [positif, negatif]
[[3 1]
 [0 4]]

              precision    recall  f1-score   support

     negatif     0.8000    1.0000    0.8889         4
     positif     1.0000    0.7500    0.8571         4

    accuracy                         0.8750         8
   macro avg     0.9000    0.8750    0.8730         8
weighted avg     0.9000    0.8750    0.8730         8



### 6.3 Tabel Perbandingan Ringkasan

In [11]:
summary = pd.DataFrame({
    'Metrik': ['Accuracy', 'Precision (positif)', 'Recall (positif)', 'F1-Score (positif)'],
    'Model Klasik': [
        accuracy_score(y_test, y_pred_classic),
        precision_score(y_test, y_pred_classic, pos_label='positif'),
        recall_score(y_test, y_pred_classic, pos_label='positif'),
        f1_score(y_test, y_pred_classic, pos_label='positif'),
    ],
    'LLM API': [
        accuracy_score(y_test, y_pred_llm),
        precision_score(y_test, y_pred_llm, pos_label='positif'),
        recall_score(y_test, y_pred_llm, pos_label='positif'),
        f1_score(y_test, y_pred_llm, pos_label='positif'),
    ],
})
summary['Metrik'] = summary['Metrik'].str.replace('(positif)', '', regex=False)
summary = summary.round(4)
summary

,Metrik,Model Klasik,LLM API
0,Accuracy,1.0,0.8750
1,Precision,1.0,1.0000
2,Recall,1.0,0.7500
3,F1-Score,1.0,0.8571


## 7. Analisis Trade-off dan Limitation

**Performa:** pada dataset ini, kedua pendekatan sama-sama mencapai skor sempurna (1.0). Namun hal ini **bukan** indikator yang menggembirakan; ini pertanda dataset terlalu mudah:

- Test set hanya **8 template unik**, dan polanya sangat jelas (kata kunci sentimen langsung terlihat).

- Model klasik dapat menghafal pola kata; LLM dapat membaca makna. Keduanya tidak benar-benar diuji.

**Trade-off dari sisi effort dan biaya:**

| Aspek | Model Klasik | LLM API |

|---|---|---|

| Training | Perlu fit dan tuning | Tidak perlu training |

| Kecepatan inference | Milidetik, offline | Sekitar 1-5 detik per review, butuh API call |

| Biaya | Hampir nol (hanya CPU) | Per token; untuk skala besar bisa signifikan |

| Maintenance | Perlu retrain jika data berubah | Cukup update prompt |

| Generalisasi bahasa | Terbatas kosakata train | Paham konteks dan bahasa gaul |

**Keterbatasan yang diamati:**

**LLM API:**

- Output kadang kosong atau tidak valid, sehingga perlu lapisan normalisasi/retry.

- Tidak 100% deterministik: suhu 0.2 mengurangi variasi, tetapi tidak menghilangkan.

- Ada biaya per panggilan dan ketergantungan ke pihak ketiga (rate limit, region, ketersediaan model).

**Model klasik:**

- Membutuhkan data berlabel yang cukup; dengan 32 data train saja sudah sangat menempel.

- Kurang peka terhadap konteks/sinonim (misal "murah tapi menipu" bisa salah dibaca).

- Jika pola ulasan berubah, model harus dilatih ulang.

## 8. Rekomendasi Technical Approach

**Untuk tahap eksperimen ini, rekomendasinya adalah memulai dari model klasik (TF-IDF + Logistic Regression) sebagai baseline produksi**, dengan alasan:

- Performa setara LLM pada dataset ini, tapi **jauh lebih murah, cepat, dan deterministik**.

- Mudah di-deploy (file kecil, tanpa ketergantungan API), cocok untuk tim kecil.

**Namun**, jika ke depannya data ulasan semakin beragam (slang, typo, sarkasme, bahasa campuran), LLM API lebih unggul karena tidak perlu dilatih ulang dan lebih memahami konteks. Strategi yang umum digunakan adalah **hybrid**: model klasik untuk volume besar yang polanya jelas, LLM untuk kasus yang membuat model klasik ragu (confidence rendah).

Keputusan akhir tetap bergantung pada budget dan kebutuhan latensi tim.